# ECG Demo — Quick Hardware Verification
Loads `ecg_demo.bit`, starts the AD7991 ADC sampler (Xilinx AXI IIC IP),
polls AXI registers for 1 second, plots DAC / RAW / Filtered waveforms.

**Pass criteria:**
- ECG_DAC range > 10 counts → DDS fix confirmed, ROM is advancing
- ECG_RAW range > 10 counts → PS-side AD7991 sampler is feeding the ADC loopback

In [ ]:
import pynq
from pynq import Overlay, MMIO
import threading
import time
import matplotlib.pyplot as plt
%matplotlib inline

BASE_ADDR = 0x43C00000
MAP_SIZE  = 0x44

ol = Overlay('/home/xilinx/jupyter_notebooks/ecg_demo.bit')
print('Overlay loaded OK')
print('IP dict keys :', list(ol.ip_dict.keys()))

In [ ]:
mmio = MMIO(BASE_ADDR, MAP_SIZE)

# Set detect threshold per algorithm_spec (reset default is 0x800, should be 2983)
mmio.write(0x38, 2983)
print(f'DETECT_THRESHOLD set to 2983 (0xBA7)')
print(f'BPM_CH_A = {mmio.read(0x00) & 0xFF} BPM')

In [ ]:
# Start the AD7991 sampler in a background thread.
# This used to live in pl/i2c_adc_driver.v (custom RTL) — that driver
# never made the chip ACK, so the I2C path moved to PS using the Xilinx
# AXI IIC IP. The sampler writes 12-bit samples into ECG_RAW (0x28) at
# 360 Hz; the FIR + R-peak in PL pick them up via adc_valid pulses.

AD7991_ADDR    = 0x28
AD7991_CFG_CH0 = 0x10
SAMPLE_RATE_HZ = 360

iic_drv = ol.axi_iic_0   # Xilinx AXI IIC IP at 0x41600000

sampler_stop = threading.Event()

def _sampler():
    period = 1.0 / SAMPLE_RATE_HZ
    next_t = time.monotonic()
    err_logged = False
    while not sampler_stop.is_set():
        try:
            iic_drv.send(AD7991_ADDR, [AD7991_CFG_CH0], 1)
            d = iic_drv.receive(AD7991_ADDR, 2)
            raw = ((d[0] & 0x0F) << 8) | (d[1] & 0xFF)
            mmio.write(0x28, raw)
        except Exception as e:
            if not err_logged:
                print(f'AD7991 sampler error: {e}')
                err_logged = True
        next_t += period
        delta = next_t - time.monotonic()
        if delta > 0:
            time.sleep(delta)
        else:
            next_t = time.monotonic()

sampler_thread = threading.Thread(target=_sampler, daemon=True)
sampler_thread.start()
time.sleep(0.05)   # let it produce a few samples before polling
print('AD7991 sampler running')

In [ ]:
N_SAMPLES = 360          # 1 second at 360 Hz
SLEEP_S   = 1.0 / 360

dac_vals  = []
raw_vals  = []
filt_vals = []

print('Sampling...')
for _ in range(N_SAMPLES):
    dac_vals.append( mmio.read(0x40) & 0xFFF)
    raw_vals.append( mmio.read(0x28) & 0xFFF)
    filt_vals.append(mmio.read(0x2C) & 0xFFF)
    time.sleep(SLEEP_S)

bpm_out = mmio.read(0x30) & 0xFF
status  = mmio.read(0x3C) & 0x3

print(f'Done.')
print(f'  BPM_OUT        : {bpm_out}')
print(f'  STATUS         : 0b{status:02b}  (signal_present={status & 1})')
print(f'  ECG_DAC range  : {max(dac_vals) - min(dac_vals)} counts  (min={min(dac_vals):#05x} max={max(dac_vals):#05x})')
print(f'  ECG_RAW range  : {max(raw_vals) - min(raw_vals)} counts  (min={min(raw_vals):#05x} max={max(raw_vals):#05x})')

dac_range = max(dac_vals) - min(dac_vals)
raw_range = max(raw_vals) - min(raw_vals)
print()
print('ECG_DAC: ' + ('PASS — DDS advancing through ROM' if dac_range > 10 else 'FAIL — DDS frozen, bitstream may be stale'))
print('ECG_RAW: ' + ('PASS — AXI IIC sampler is feeding ADC values' if raw_range > 10 else 'WARN — ADC flat, check PMOD AD2 wiring on JB or sampler errors above'))

In [ ]:
# Stop the sampler before plotting (clean shutdown).
sampler_stop.set()
sampler_thread.join(timeout=1.0)
print('Sampler stopped.')

In [ ]:
t_ms = [i * (1000.0 / 360) for i in range(N_SAMPLES)]

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
fig.suptitle('ECG Demo — Quick Verification', fontsize=14)

axes[0].plot(t_ms, dac_vals, color='steelblue', linewidth=0.9)
axes[0].set_ylabel('Counts')
axes[0].set_title('ECG_DAC — DDS ROM output (Ch A)')

axes[1].plot(t_ms, raw_vals, color='darkorange', linewidth=0.9)
axes[1].set_ylabel('Counts')
axes[1].set_title('ECG_RAW — ADC loopback (PS → AXI IIC → AD7991)')

axes[2].plot(t_ms, filt_vals, color='seagreen', linewidth=0.9)
axes[2].set_ylabel('Counts')
axes[2].set_title('ECG_FILTERED — FIR output')
axes[2].set_xlabel('Time (ms)')

plt.tight_layout()
plt.savefig('/home/xilinx/jupyter_notebooks/ecg_quick_test.png', dpi=120)
plt.show()
print('Plot saved to ecg_quick_test.png')